In [ ]:
# STEP 1: Upload the file
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# Replace 'error_bad_lines' with 'on_bad_lines'
df = pd.read_csv('/content/amazon.csv', header=None, on_bad_lines='skip', sep=',')

# Print some info to check the data
print(df.head())  # Print

                                                   0  \
0  __label__2 Great CD: My lovely Pat has one of ...   
1  __label__2 One of the best game music soundtra...   
2  __label__1 Batteries died within a year ...: I...   
3                              __label__2 works fine   
4  __label__2 Great for the non-audiophile: Revie...   

                                                   1  \
0                                    no matter black   
1   the music I heard (plus the connection to Chr...   
2                                 after about a year   
3   but Maha Energy is better: Check out Maha Ene...   
4   but don't want to replace them with DVD's. Th...   

                                                   2  \
0                                              white   
1   and it remains one of my favorite albums. The...   
2   the batteries would not hold a charge. Might ...   
3   with option for slower charge (better for bat...   
4   easy to setup and resolution and special e

In [ ]:
import pandas as pd
import re
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils import resample

# Load dataset
df_raw = pd.read_csv('/content/amazon.csv', header=None, names=["Text"], on_bad_lines='skip')

# Better keyword-based labeling
def guess_label(text):
    text = str(text).lower()
    if any(word in text for word in ['worst', 'bad', 'terrible', 'hate', 'awful','waste','useless','sad','not']):
        return 0  # Negative
    elif any(word in text for word in ['love', 'great', 'excellent', 'best', 'amazing','good','awesome','useful']):
        return 1  # Positive
    else:
        return -1  # Ambiguous → discard

df_raw['Label'] = df_raw['Text'].apply(guess_label)
df_raw = df_raw[df_raw['Label'] != -1]  # remove uncertain samples

# Clean text
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'@[\w_]+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text.strip()

df_raw['clean_text'] = df_raw['Text'].apply(clean_text)

# Check balance
print("\nLabel distribution:")
print(df_raw['Label'].value_counts())

# Balance the dataset
df_pos = df_raw[df_raw['Label'] == 1]
df_neg = df_raw[df_raw['Label'] == 0]
min_len = min(len(df_pos), len(df_neg))

df_balanced = pd.concat([
    resample(df_pos, n_samples=min_len, random_state=42),
    resample(df_neg, n_samples=min_len, random_state=42)
]).sample(frac=1, random_state=42)

# Prepare data
X = df_balanced['clean_text']
y = df_balanced['Label']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train Naive Bayes
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

#Evaluate
y_pred = clf.predict(X_test_vec)
print("\nEvaluation:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Predict function
def predict_sentiment(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])
    pred = clf.predict(vector)[0]
    return "Positive" if pred == 1 else "Negative"

# Sample predictions
print("\nSample Predictions:")
sample_texts = [
    "I love this product!",
    "Worst thing I've ever bought.",
    "Not bad, just okay.",
    "Exceeded my expectations!",
    "Terrible quality."
]

for text in sample_texts:
    print(f"{text} --> Prediction: {predict_sentiment(text)}")

# User input
print("\nTry your own review (type 'e' to quit):")
while True:
    user_input = input("Enter a review: ")
    if user_input.lower() == 'e':
        print("Exited.")
        break
    print("Prediction:", predict_sentiment(user_input))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
<ipython-input-4-0b55b72f08d7>:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_raw['clean_text'] = df_raw['Text'].apply(clean_text)



Label distribution:
Label
0    1998
1    1173
Name: count, dtype: int64

Evaluation:
Accuracy: 0.8787234042553191
Confusion Matrix:
 [[193  30]
 [ 27 220]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.87      0.87       223
           1       0.88      0.89      0.89       247

    accuracy                           0.88       470
   macro avg       0.88      0.88      0.88       470
weighted avg       0.88      0.88      0.88       470


Sample Predictions:
I love this product! --> Prediction: Positive
Worst thing I've ever bought. --> Prediction: Negative
Not bad, just okay. --> Prediction: Negative
Exceeded my expectations! --> Prediction: Negative
Terrible quality. --> Prediction: Negative

Try your own review (type 'e' to quit):
Enter a review: not worth buying
Prediction: Negative
Enter a review: amazing product
Prediction: Positive
Enter a review: stinky product
Prediction: Negative
Enter a review: hat it
Predicti